In [10]:
import pandas as pd
import os
import sys
import seaborn as sns
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
sys.path.append(os.path.abspath(".."))
from scripts.utils import make_engine
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [11]:
engine = make_engine()

# read SQL query
BASE_DIR = Path.cwd().parent
sql_query = BASE_DIR / "queries" / "rfm_prep.sql"
with open(sql_query, "r") as file:
    sql_query = file.read()

# get df
df_rfm = pd.read_sql_query(sql_query, con=engine)

df_rfm.head()

,customer_id,recency,frequency,monetary
0,C0001,587,1,997.50
1,C0002,93,2,1226.30
2,C0003,374,3,1854.30
3,C0004,92,1,713.05
4,C0005,589,1,2047.25


In [14]:
# Create recency Scoring
r_bins = [-1, 60, 90, 150, 365, float('inf')]
r_labels = [5, 4, 3, 2, 1] 
df_rfm['R_Score'] = pd.cut(df_rfm['recency'], bins=r_bins, labels=r_labels).astype(int)

# Create Frequency Scoring
f_bins = [0, 1, 2, 3, 4, float('inf')]
f_labels = [5, 4, 3, 2, 1]
df_rfm["F_Score"] = pd.cut(df_rfm["frequency"], bins=f_bins, labels=f_labels).astype(int)

# Create Monetary Scoring
m_bins = [0, 1000, 2500, 4000, 5550, float('inf')]
m_labels = [5, 4, 3, 2, 1]
df_rfm["M_Score"] = pd.cut(df_rfm["monetary"], bins=m_bins, labels=f_labels).astype(int)

df_rfm

,customer_id,recency,frequency,monetary,R_Score,F_Score,M_Score
0,C0001,587,1,997.50,1,5,5
1,C0002,93,2,1226.30,3,4,4
2,C0003,374,3,1854.30,1,3,4
3,C0004,92,1,713.05,3,5,5
4,C0005,589,1,2047.25,1,5,4
...,...,...,...,...,...,...,...
433,C0496,43,3,3373.50,5,3,3
434,C0497,502,1,948.10,1,5,5
435,C0498,362,2,2727.20,2,4,3
436,C0499,139,1,423.25,3,5,5


In [16]:
# assign segments based on R_Score and F_Score
def assign_segment(row):
    r = row["R_Score"]
    f = row["F_Score"]
    m = row["M_Score"]

    if r > 4: 
        if f >= 4 and m >= 4:
            return "Champions"
        elif f >= 4 and m >= 3:
            return "Core Enthusiast"
        elif f >= 3:
            return "Potential Loyalist"
        else:
            return "Recent Samplers"
        
    elif r == 4 and f >= 2:
        return "Routine Restock"
    
    elif r >= 3:
        if f >= 4:
            return "Loyal Customers"
        else:
            return "About to Sleep"
        
    elif r >= 2 and m >= 4:
        return "High-Priority Win-Backs"
    
    elif r == 1 and f == 1 and m == 1:
        return "Lost Customers"
    
    else:
        return "Needs Attention"
    
df_rfm["Segment"] = df_rfm.apply(assign_segment, axis=1)

df_rfm["Segment"].value_counts()

Segment
Needs Attention            178
High-Priority Win-Backs     82
Routine Restock             56
About to Sleep              35
Loyal Customers             30
Champions                   26
Recent Samplers             15
Potential Loyalist          11
Core Enthusiast              4
Lost Customers               1
Name: count, dtype: int64